# British Airways Customer Booking Prediction

This notebook explores the supplied customer booking data and builds a Random Forest model to predict whether a customer completes a booking. The data file is not included in this repository. Place `customer_booking.csv` beside this notebook before running it.

## 1. Import packages and load the data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)

DATA_PATH = Path("customer_booking.csv")
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

## 2. Explore the dataset

The target column is `booking_complete`: 1 means the customer completed a booking, and 0 means they did not. The booking classes are imbalanced, so accuracy alone is not enough to judge the model.

In [ ]:
df.info()
df["booking_complete"].value_counts(dropna=False)

In [ ]:
booking_rate = df["booking_complete"].mean() * 100
print(f"Completed booking rate: {booking_rate:.1f}%")
df.describe(include="all").T.head(20)

## 3. Prepare the features

Convert the weekday names to numbers. Keep the target separate from the input features. Text columns will be one-hot encoded inside a scikit-learn pipeline so the same transformation is applied during validation and testing.

In [ ]:
day_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
df["flight_day"] = df["flight_day"].map(day_map)

X = df.drop(columns="booking_complete")
y = df["booking_complete"]
categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical columns:", categorical_features)
print("Numeric columns:", numeric_features)
print("Missing values in features:", int(X.isna().sum().sum()))

## 4. Split the data and define the model

Hold out 20% of the records for a final test. `stratify=y` keeps the proportion of completed bookings similar in both sets. The model gives extra weight to the less common booking class.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1
    )),
])
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")

## 5. Evaluate with 5-fold cross-validation

Cross-validation trains and evaluates the pipeline five times using different parts of the training data. The averages give a more dependable view than a single split.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1
)
cv_summary = pd.DataFrame({
    "mean": {metric: cv_results[f"test_{metric}"].mean() for metric in scoring},
    "std": {metric: cv_results[f"test_{metric}"].std() for metric in scoring},
})
(cv_summary * 100).round(1).rename(index={"roc_auc": "ROC-AUC"})

## 6. Test the final model on the held-out data

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}
pd.Series(test_metrics).mul(100).round(1).rename("Test set (%)")

In [ ]:
print(classification_report(y_test, y_pred, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", values_format="d")
plt.title("Held-out test set")
plt.tight_layout()
plt.show()

## 7. See which original variables the model relied on

One-hot encoding creates multiple columns for text features. Sum their importance scores back to the original variable so the results are easier to read. Importance shows how much the fitted model used each variable; it does not prove that the variable causes a booking.

In [ ]:
fitted_preprocessor = model.named_steps["preprocessor"]
encoded_names = fitted_preprocessor.get_feature_names_out()
raw_importances = model.named_steps["classifier"].feature_importances_

def original_feature(encoded_name):
    name = encoded_name.split("__", 1)[-1]
    for feature in sorted(categorical_features, key=len, reverse=True):
        if name == feature or name.startswith(feature + "_"):
            return feature
    for feature in numeric_features:
        if name == feature:
            return feature
    return name

importance = pd.DataFrame({
    "feature": [original_feature(name) for name in encoded_names],
    "importance": raw_importances,
})
importance = importance.groupby("feature", as_index=False)["importance"].sum()
importance = importance.sort_values("importance", ascending=False)
importance

In [ ]:
top_features = importance.head(10).sort_values("importance")
plt.figure(figsize=(9, 5))
plt.barh(top_features["feature"], top_features["importance"], color="#b20d2f")
plt.xlabel("Aggregated Random Forest feature importance")
plt.title("Variables the model relied on most")
plt.tight_layout()
plt.show()

## 8. Findings and limitations

- Compare precision and recall alongside accuracy because completed bookings are a minority class.
- Low recall means many actual bookers are missed at the model's default decision threshold.
- Try different thresholds and compare the number of customers found with the false-positive rate before using scores for marketing.
- Route has many distinct values. Its feature importance may be affected by this high cardinality, so validate it with permutation importance or another method.
- This is a learning project. Do not use it to contact customers or make operational decisions without further validation.